In [2]:
# 데이터 다운로드
from roboflow import Roboflow

rf = Roboflow(api_key="Aov0WcVV5Y6GR71aEhV6")
project = rf.workspace("6rainstorm-yqytq").project("6rainstorm-final-project")
version = project.version(6)
dataset = version.download("yolov8", location="./solar_data")

loading Roboflow workspace...
loading Roboflow project...


In [1]:
# 클래스 매핑 
import yaml, os, shutil

CLASS_MAP = {
    0: 3, 1: 4, 2: 1,
    3: 0, 4: 4, 5: 2,
}
OUR_CLASSES = {
    0: 'normal', 1: 'dust', 2: 'snow',
    3: 'bird_dropping', 4: 'physical_damage',
}

src_base = './solar_data'
dst_base = './solar_data_mapped'

splits = {'train': 'train', 'valid': 'val', 'test': 'test'}

for src_split, dst_split in splits.items():
    src_label_dir = f'{src_base}/{src_split}/labels'
    src_image_dir = f'{src_base}/{src_split}/images'
    dst_label_dir = f'{dst_base}/{dst_split}/labels'
    dst_image_dir = f'{dst_base}/{dst_split}/images'

    os.makedirs(dst_label_dir, exist_ok=True)
    os.makedirs(dst_image_dir, exist_ok=True)

    for img in os.listdir(src_image_dir):
        shutil.copy(f'{src_image_dir}/{img}', f'{dst_image_dir}/{img}')

    for label_file in os.listdir(src_label_dir):
        src_path = f'{src_label_dir}/{label_file}'
        dst_path = f'{dst_label_dir}/{label_file}'
        with open(src_path, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            new_lines.append(f"{CLASS_MAP[int(parts[0])]} {' '.join(parts[1:])}\n")
        with open(dst_path, 'w') as f:
            f.writelines(new_lines)
    print(f"[{src_split} → {dst_split}] 변환 완료")

new_yaml = {
    'path': os.path.abspath('./solar_data_mapped'),  # 절대경로로 변환
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc': 5,
    'names': {0:'normal', 1:'dust', 2:'snow',
              3:'bird_dropping', 4:'physical_damage'}
}
with open('./solar_data_mapped/dataset.yaml', 'w') as f:
    yaml.dump(new_yaml, f)
print("dataset.yaml 생성 완료!")

[train → train] 변환 완료
[valid → val] 변환 완료
[test → test] 변환 완료
dataset.yaml 생성 완료!


In [3]:
import os
print(os.path.exists('./solar_data_mapped'))
print(os.listdir('./solar_data_mapped') if os.path.exists('./solar_data_mapped') else "폴더 없음")

True
['dataset.yaml', 'test', 'train', 'val']


In [4]:
from ultralytics import YOLO
import torch

print(f"GPU 사용 가능: {torch.cuda.is_available()}")
print(f"GPU 이름: {torch.cuda.get_device_name()}")

# yolov11s로 변경
model = YOLO('yolo11s.pt')

results = model.train(
    data='./solar_data_mapped/dataset.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    name='solar_panel_v3_yolo11s',  
    patience=30,
    device=0,
    cls=2.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
    save=True,
    save_period=10,
    project='./weights',
)

print(f"mAP50: {results.results_dict['metrics/mAP50(B)']:.4f}")

GPU 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 3060 Ti
New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.10.20 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./solar_data_mapped/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yo

In [5]:
from ultralytics import YOLO
import time
import torch

model = YOLO(r'C:\Users\hp112\OneDrive\바탕 화면\minji\pannel\runs\detect\weights\solar_panel_v3_yolo11s\weights\best.pt')

m = model.val(data='./solar_data_mapped/dataset.yaml', verbose=False)

# FPS 측정
dummy = torch.zeros(1, 3, 640, 640).to('cuda')
start = time.time()
for _ in range(100):
    model.predict(dummy, verbose=False)
fps = 100 / (time.time() - start)

class_names = ['normal', 'dust', 'snow', 'bird_dropping', 'physical_damage']

print(f"\n{'='*45}")
print(f"[YOLOv11s]")
print(f"  mAP50:    {m.box.map50:.4f}")
print(f"  mAP50-95: {m.box.map:.4f}")
print(f"  Recall:   {m.box.r.mean():.4f}")
print(f"  FPS:      {fps:.1f}")
print(f"\n  클래스별 AP50 / Recall:")
for i, name in enumerate(class_names):
    ap = m.box.ap50[i]
    recall = m.box.r[i]
    status = "나쁨" if ap < 0.3 else "보통" if ap < 0.5 else "양호"
    print(f"    {name:20s}: AP={ap:.3f} | Recall={recall:.3f} {status}")

Ultralytics 8.4.37  Python-3.10.20 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
YOLO11s summary (fused): 101 layers, 9,414,735 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 194.239.9 MB/s, size: 66.7 KB)
val: Scanning C:\Users\hp112\OneDrive\바탕 화면\minji\pannel\solar_data_mapped\val\labels.cache... 409 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 409/409  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 1304, len(boxes) = 1560. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 5.7it/s 4.6s0.1s
                   all        409       1560      0.779      0.685      0.712       0.51
Speed: 3.1ms preprocess, 4.8ms inference, 0.0ms loss, 0.6ms postprocess per image
Results